# Enhanced Bedrock Experimentation Notebook

**Purpose**: Experiment with LLM context and prompts using AWS Bedrock

**Enhancements**: Error handling, quota docs, parameter explanations, troubleshooting

**Features**:
- Easy model switching (Claude / Nova)
- Load context from multiple sources (including Confluence exports)
- Track experiments (model, tokens, cost)
- Robust error handling with retry logic
- Cost budget enforcement

---
## SECTION 1: SETUP & IMPORTS

In [ ]:
import boto3
import json
import pandas as pd
from datetime import datetime
from typing import Optional, Dict, Any
from pathlib import Path
import time
import random
from botocore.exceptions import ClientError

# Create experiments directory
Path("experiments").mkdir(exist_ok=True)

print("✓ Imports successful")

---
## SECTION 2: MODEL CONFIGURATION & QUOTAS

In [ ]:
# Available models in Sydney (ap-southeast-2)
MODELS = {
    # Claude (Anthropic)
    "claude-haiku": "anthropic.claude-3-haiku-20240307-v1:0",
    "claude-sonnet": "anthropic.claude-3-sonnet-20240229-v1:0",
    "claude-sonnet-3.5": "anthropic.claude-3-5-sonnet-20241022-v2:0",
    # Nova (Amazon)
    "nova-micro": "amazon.nova-micro-v1:0",
    "nova-lite": "amazon.nova-lite-v1:0",
    "nova-pro": "amazon.nova-pro-v1:0",
}

# Pricing per 1M tokens (USD)
PRICING = {
    "anthropic.claude-3-haiku-20240307-v1:0": {"input": 0.25, "output": 1.25},
    "anthropic.claude-3-sonnet-20240229-v1:0": {"input": 3.00, "output": 15.00},
    "anthropic.claude-3-5-sonnet-20241022-v2:0": {"input": 3.00, "output": 15.00},
    "amazon.nova-micro-v1:0": {"input": 0.035, "output": 0.14},
    "amazon.nova-lite-v1:0": {"input": 0.06, "output": 0.24},
    "amazon.nova-pro-v1:0": {"input": 0.80, "output": 3.20},
}

# Context window limits (INPUT tokens - how much you can send TO the model)
CONTEXT_LIMITS = {
    "anthropic.claude-3-haiku-20240307-v1:0": 200_000,
    "anthropic.claude-3-sonnet-20240229-v1:0": 200_000,
    "anthropic.claude-3-5-sonnet-20241022-v2:0": 200_000,
    "amazon.nova-micro-v1:0": 128_000,
    "amazon.nova-lite-v1:0": 300_000,
    "amazon.nova-pro-v1:0": 300_000,
}

# Output token limits (OUTPUT tokens - how much the model can generate back)
OUTPUT_LIMITS = {
    "anthropic.claude-3-haiku-20240307-v1:0": 4_096,
    "anthropic.claude-3-sonnet-20240229-v1:0": 4_096,
    "anthropic.claude-3-5-sonnet-20241022-v2:0": 8_192,
    "amazon.nova-micro-v1:0": 5_000,
    "amazon.nova-lite-v1:0": 5_000,
    "amazon.nova-pro-v1:0": 5_000,
}

# AWS Bedrock Quotas (as of setup - verify in AWS Console)
QUOTAS = {
    "requests_per_minute": {
        "claude-haiku": 10_000,
        "claude-sonnet": 10_000,
        "claude-sonnet-3.5": 10_000,
        "nova-micro": 10_000,
        "nova-lite": 10_000,
        "nova-pro": 10_000,
    },
    "tokens_per_minute": {
        "claude-haiku": 300_000,
        "claude-sonnet": 300_000,
        "claude-sonnet-3.5": 300_000,
        "nova-micro": 300_000,
        "nova-lite": 300_000,
        "nova-pro": 300_000,
    }
}

print("✓ Model configuration loaded")
print(f"  Available models: {list(MODELS.keys())}")

---
## SECTION 3: UNDERSTANDING KEY CONCEPTS

### 🎓 Context Window vs Max Tokens

```
┌─────────────────────────────────────────────────────────────┐
│                    Context Window (INPUT)                   │
│  "How much you can send TO the model"                       │
│  Example: Claude Haiku = 200,000 tokens                     │
│  ≈ 150,000 words ≈ 300 pages of text                       │
└─────────────────────────────────────────────────────────────┘
                              ↓
                    ┌──────────────────┐
                    │   LLM Processing │
                    └──────────────────┘
                              ↓
┌─────────────────────────────────────────────────────────────┐
│                   Max Tokens (OUTPUT)                       │
│  "How much the model can generate back"                    │
│  Example: Claude Haiku = 4,096 tokens                       │
│  ≈ 3,000 words ≈ 6-8 pages of text                         │
└─────────────────────────────────────────────────────────────┘
```

**Why the asymmetry?**
- READING (input): Single pass, fast, cheap
- GENERATING (output): Token-by-token, slow, expensive
- Each output token requires ~1000x more computation than input

**Real example from business rules:**
- Input: 38,000 tokens (entire business rules corpus)
- Output: 2,500 tokens (restructured tables)
- Total: 41,019 tokens charged

**💰 Cost implications:**
- Input tokens are CHEAPER per token
- Output tokens are MORE EXPENSIVE per token
- Always consider this when designing prompts!

---
## SECTION 4: ENHANCED BEDROCK CLIENT WITH ERROR HANDLING

In [ ]:
class BedrockClient:
    """
    Unified client for Bedrock models (Claude and Nova) with robust error handling.
    
    Key enhancements:
    - Automatic retry with exponential backoff
    - Rate limit handling
    - Cost tracking
    - Detailed error messages
    """
    
    def __init__(self, model: str = "claude-haiku", region: str = "ap-southeast-2"):
        self.client = boto3.client("bedrock-runtime", region_name=region)
        self.model_id = MODELS.get(model, model)
        self.model_alias = model if model in MODELS else self._get_alias(self.model_id)
        self.provider = "anthropic" if self.model_id.startswith("anthropic.") else "amazon"
        
        # Validate model exists
        if model not in MODELS and model not in MODELS.values():
            raise ValueError(
                f"Unknown model: {model}. "
                f"Available: {list(MODELS.keys())}"
            )
    
    def _get_alias(self, model_id: str) -> str:
        for alias, mid in MODELS.items():
            if mid == model_id:
                return alias
        return model_id
    
    def _build_body(self, prompt: str, system: Optional[str], 
                    max_tokens: int, temperature: float) -> dict:
        if self.provider == "anthropic":
            body = {
                "anthropic_version": "bedrock-2023-05-31",
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": [{"role": "user", "content": prompt}]
            }
            if system:
                body["system"] = system
        else:  # Amazon Nova
            body = {
                "inferenceConfig": {"maxTokens": max_tokens, "temperature": temperature},
                "messages": [{"role": "user", "content": [{"text": prompt}]}]
            }
            if system:
                body["system"] = [{"text": system}]
        return body
    
    def _parse_response(self, result: dict) -> dict:
        if self.provider == "anthropic":
            content = result["content"][0]["text"]
            usage = result.get("usage", {})
            input_tokens = usage.get("input_tokens", 0)
            output_tokens = usage.get("output_tokens", 0)
        else:  # Amazon Nova
            content = result["output"]["message"]["content"][0]["text"]
            usage = result.get("usage", {})
            input_tokens = usage.get("inputTokens", 0)
            output_tokens = usage.get("outputTokens", 0)
        
        return {
            "content": content,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens": input_tokens + output_tokens,
        }
    
    def _calculate_cost(self, input_tokens: int, output_tokens: int) -> float:
        pricing = PRICING.get(self.model_id, {"input": 0, "output": 0})
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        return input_cost + output_cost
    
    def invoke(self, prompt: str, system: Optional[str] = None,
               max_tokens: int = 4096, temperature: float = 0.0,
               max_retries: int = 3) -> dict:
        """
        Send prompt to model and return response with usage stats.
        
        Args:
            prompt: User prompt text
            system: System prompt (optional)
            max_tokens: Maximum OUTPUT tokens (NOT input!)
            temperature: Randomness (0.0 = deterministic, 1.0 = creative)
            max_retries: Number of retry attempts on throttling
        
        Returns:
            dict with keys: content, input_tokens, output_tokens, total_tokens, 
                           model, model_id, cost
        """
        # Validate max_tokens doesn't exceed model limit
        max_output = OUTPUT_LIMITS.get(self.model_id, 4096)
        if max_tokens > max_output:
            print(f"⚠️  Warning: max_tokens={max_tokens} exceeds {self.model_alias} "
                  f"limit of {max_output}. Capping to {max_output}.")
            max_tokens = max_output
        
        # Retry logic with exponential backoff
        for attempt in range(max_retries):
            try:
                response = self.client.invoke_model(
                    modelId=self.model_id,
                    contentType="application/json",
                    accept="application/json",
                    body=json.dumps(self._build_body(prompt, system, max_tokens, temperature))
                )
                
                result = self._parse_response(json.loads(response["body"].read()))
                result["model"] = self.model_alias
                result["model_id"] = self.model_id
                result["cost"] = self._calculate_cost(result["input_tokens"], result["output_tokens"])
                
                return result
                
            except ClientError as e:
                error_code = e.response['Error']['Code']
                error_msg = e.response['Error']['Message']
                
                if error_code == 'ThrottlingException':
                    if attempt < max_retries - 1:
                        wait_time = (2 ** attempt) + random.uniform(0, 1)
                        print(f"⚠️  Rate limited. Retrying in {wait_time:.1f}s (attempt {attempt + 1}/{max_retries})...")
                        time.sleep(wait_time)
                        continue
                    else:
                        raise Exception(
                            f"Rate limit exceeded after {max_retries} retries. "
                            f"Try: (1) Reduce requests/min, (2) Add delays between calls, "
                            f"(3) Request quota increase in AWS Console."
                        )
                
                elif error_code == 'ValidationException':
                    raise ValueError(
                        f"Invalid request: {error_msg}\n"
                        f"Check: (1) max_tokens within limit, (2) prompt format, (3) temperature range"
                    )
                
                elif error_code == 'AccessDeniedException':
                    raise PermissionError(
                        f"Access denied: {error_msg}\n"
                        f"Check: (1) IAM permissions (bedrock:InvokeModel), "
                        f"(2) Model access enabled in Bedrock console, "
                        f"(3) Anthropic marketplace subscription"
                    )
                
                else:
                    raise
    
    def get_context_limit(self) -> int:
        return CONTEXT_LIMITS.get(self.model_id, 200_000)
    
    def get_output_limit(self) -> int:
        return OUTPUT_LIMITS.get(self.model_id, 4_096)
    
    def __repr__(self):
        return (f"BedrockClient(model='{self.model_alias}', provider='{self.provider}', "
                f"context={self.get_context_limit():,}, max_output={self.get_output_limit():,})")

# Factory function for easy client creation
def get_client(model: str = "claude-haiku") -> BedrockClient:
    """
    Get a Bedrock client for the specified model.
    
    Available models:
        Claude: 'claude-haiku', 'claude-sonnet', 'claude-sonnet-3.5'
        Nova:   'nova-micro', 'nova-lite', 'nova-pro'
    """
    return BedrockClient(model=model)

print("✓ BedrockClient class loaded with error handling")

---
## SECTION 5: ENHANCED CONTEXT UTILITIES

In [ ]:
def load_text_file(path: str) -> str:
    """Load text from file (.txt, .md, etc.)"""
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

def load_markdown_file(path: str) -> str:
    """Load markdown file (alias for load_text_file for clarity)."""
    return load_text_file(path)

def load_confluence_doc(path: str) -> str:
    """
    Parse Confluence's fake .doc export (MIME-encoded HTML).
    
    Requires: pip install beautifulsoup4
    """
    import email
    import quopri
    from bs4 import BeautifulSoup
    
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Parse as MIME message
    msg = email.message_from_string(content)
    
    # Extract HTML part
    html_content = ""
    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() == "text/html":
                payload = part.get_payload(decode=False)
                html_content = quopri.decodestring(payload.encode()).decode('utf-8', errors='ignore')
                break
    else:
        html_content = msg.get_payload(decode=True).decode('utf-8', errors='ignore')
    
    # Strip HTML tags, keep text
    soup = BeautifulSoup(html_content, 'html.parser')
    
    # Extract text with basic structure
    lines = []
    for element in soup.find_all(['h1', 'h2', 'h3', 'h4', 'p', 'li', 'td', 'th']):
        tag = element.name
        text = element.get_text(strip=True)
        if not text:
            continue
        if tag == 'h1':
            lines.append(f"\n# {text}")
        elif tag == 'h2':
            lines.append(f"\n## {text}")
        elif tag == 'h3':
            lines.append(f"\n### {text}")
        elif tag == 'h4':
            lines.append(f"\n#### {text}")
        elif tag == 'li':
            lines.append(f"- {text}")
        else:
            lines.append(text)
    
    return "\n".join(lines)

def load_excel_as_text(path: str, sheet_name: str = None) -> str:
    """Load Excel file and convert to text format."""
    df = pd.read_excel(path, sheet_name=sheet_name)
    return df.to_string()

def load_csv_as_text(path: str) -> str:
    """Load CSV file and convert to text format."""
    df = pd.read_csv(path)
    return df.to_string()

def estimate_tokens(text: str) -> int:
    """Rough token estimate (~4 chars per token for English)."""
    return len(text) // 4

def context_summary(text: str, name: str = "Context") -> int:
    """Print summary of context size and return estimated token count."""
    chars = len(text)
    tokens_est = estimate_tokens(text)
    lines = text.count('\n') + 1
    
    print(f"\n{name}:")
    print(f"  Characters: {chars:,}")
    print(f"  Lines: {lines:,}")
    print(f"  Est. tokens: ~{tokens_est:,}")
    
    # Warn if approaching context limits
    if tokens_est > 150_000:
        print(f"  ⚠️  WARNING: Approaching 200K context limit for most models!")
    elif tokens_est > 100_000:
        print(f"  ⚠️  Large context - consider Nova models (300K limit)")
    
    return tokens_est

print("✓ Context utilities loaded (including Confluence doc parser)")

---
## SECTION 6: ENHANCED EXPERIMENT TRACKER WITH BUDGET CONTROL

In [ ]:
class ExperimentTracker:
    """
    Track experiments and save results with optional budget enforcement.
    
    Enhancements:
    - Cost budget limits
    - Warning thresholds
    - Detailed token breakdown
    """
    
    def __init__(self, name: str, budget_usd: float = None):
        self.name = name
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiments = []
        self.budget_usd = budget_usd
        self._warned_at_threshold = False
    
    def log(self, response: dict, prompt_preview: str = "", notes: str = ""):
        """Log an experiment result."""
        cost = response.get("cost", 0)
        
        # Check budget
        if self.budget_usd is not None:
            total_cost = self.total_cost() + cost
            remaining = self.budget_usd - total_cost
            
            # Block if over budget
            if remaining < 0:
                raise ValueError(
                    f"💰 BUDGET EXCEEDED!\n"
                    f"   Budget: ${self.budget_usd:.4f}\n"
                    f"   Current total: ${self.total_cost():.4f}\n"
                    f"   This request: ${cost:.4f}\n"
                    f"   Would exceed by: ${abs(remaining):.4f}\n"
                    f"   Increase budget or stop experimenting."
                )
            
            # Warn at 80% threshold
            if remaining < (self.budget_usd * 0.2) and not self._warned_at_threshold:
                print(f"⚠️  WARNING: 80% of budget used (${total_cost:.4f} / ${self.budget_usd:.4f})")
                self._warned_at_threshold = True
        
        # Log experiment
        self.experiments.append({
            "timestamp": datetime.now().isoformat(),
            "model": response.get("model", ""),
            "input_tokens": response.get("input_tokens", 0),
            "output_tokens": response.get("output_tokens", 0),
            "total_tokens": response.get("total_tokens", 0),
            "cost_usd": cost,
            "prompt_preview": prompt_preview[:100] + "..." if len(prompt_preview) > 100 else prompt_preview,
            "response_preview": response.get("content", "")[:200] + "..." if len(response.get("content", "")) > 200 else response.get("content", ""),
            "notes": notes,
        })
        
        print(f"✓ Logged: {response.get('model')} | "
              f"{response.get('input_tokens'):,} in / {response.get('output_tokens'):,} out | "
              f"${cost:.6f}")
    
    def summary(self) -> pd.DataFrame:
        """Return summary as DataFrame."""
        return pd.DataFrame(self.experiments)
    
    def total_cost(self) -> float:
        """Calculate total cost across all experiments."""
        return sum(e["cost_usd"] for e in self.experiments)
    
    def total_tokens(self) -> int:
        """Calculate total tokens across all experiments."""
        return sum(e["total_tokens"] for e in self.experiments)
    
    def budget_status(self):
        """Print budget usage summary."""
        total = self.total_cost()
        if self.budget_usd is not None:
            pct = (total / self.budget_usd) * 100
            remaining = self.budget_usd - total
            print(f"\n💰 Budget Status:")
            print(f"   Spent: ${total:.4f} ({pct:.1f}%)")
            print(f"   Remaining: ${remaining:.4f}")
            print(f"   Total budget: ${self.budget_usd:.4f}")
        else:
            print(f"\n💰 Total spent: ${total:.4f} (no budget set)")
    
    def save(self, path: str = None):
        """Save experiments to CSV."""
        if path is None:
            path = f"experiments/{self.name}_{self.timestamp}.csv"
        df = self.summary()
        df.to_csv(path, index=False)
        
        print(f"\n📊 Experiment Summary:")
        print(f"   Saved to: {path}")
        print(f"   Total experiments: {len(self.experiments)}")
        print(f"   Total tokens: {self.total_tokens():,}")
        print(f"   Total cost: ${self.total_cost():.4f}")
        self.budget_status()
        
        return path

print("✓ Enhanced ExperimentTracker loaded with budget control")

---
## SECTION 7: CONNECTION TEST

In [ ]:
print("\n" + "="*70)
print("TESTING BEDROCK CONNECTIVITY")
print("="*70)

# Test with default model (Claude Haiku - cheapest)
client = get_client("claude-haiku")
print(f"\nUsing: {client}")

try:
    response = client.invoke("Tell me a joke about data scientists", temperature=0.5)
    print(f"\n✅ Connection successful!")
    print(f"   Response: {response['content']}")
    print(f"   Tokens: {response['input_tokens']} in / {response['output_tokens']} out")
    print(f"   Cost: ${response['cost']:.6f}")
except Exception as e:
    print(f"\n❌ Connection failed: {e}")

---
## SECTION 8: PARAMETER GUIDE

### 🎛️ Parameter Explanations

**1. max_tokens (Output limit)**
- Controls maximum response length
- NOT the same as context window!
- Claude Haiku: max 4,096 tokens (~3,000 words)
- Claude Sonnet 3.5: max 8,192 tokens (~6,000 words)

When to adjust:
- Short answers: 512-1024 tokens
- Medium responses: 2048 tokens
- Long analysis: 4096 tokens
- Very long: Use Sonnet 3.5 (8192 max)

**2. temperature (Randomness control)**
- Range: 0.0 to 1.0
- 0.0 = Deterministic (same input → same output)
- 1.0 = Creative/random

When to use:
- temperature=0.0: Categorization, data extraction, factual tasks
- temperature=0.3: Slightly varied responses
- temperature=0.7: Creative writing, brainstorming

**For business rules: ALWAYS use temperature=0.0 for consistency!**

**3. system (System prompt)**
- Sets the AI's role/behavior
- Optional but powerful
- Examples:
  - "You are a financial transaction categorization expert."
  - "You are a data quality analyst. Be concise and specific."

In [ ]:
# Simple list
print("Available models:", list(MODELS.keys()))

In [ ]:
BUSINESS_RULES

## 8. Multi-Turn Prompting

#### a. Setup + Define All Prompts

In [ ]:
# Load rules
rules = load_confluence_doc("assets/bank_patterns.doc")
context_summary(rules, "Business Rules")

# Define all prompts upfront
prompt_turn_1 = f"""Analyze and document ALL general patterns.

After analyzing, ADD COLUMNS that would be useful.
Always include: Spend/Non-Spend, Category Key

**General Patterns:**
| Pattern | Spend/Non-Spend | Category | [Your Columns...] | Notes |

**Transaction Type Rules:**
| Type | Spend/Non-Spend | Default Category | [Your Columns...] | Exceptions | Notes |

Instructions:
- Analyze first, choose useful columns
- Document ALL rules comprehensively

=== BUSINESS RULES ===
{rules}
===
"""

prompt_turn_2 = """Document ALL provider-specific patterns using same columns.

**CBA:**
| Pattern | Spend/Non-Spend | Category | [Same Columns...] | Why Different | Notes |

**Westpac, ANZ, NAB, Bankwest, Up, Macquarie, ING, P&N, St George, HSBC, AMP, UBANK, Suncorp, Bank Australia:**
[Continue for all providers]

Be comprehensive."""

prompt_turn_3 = """Identify ALL issues:

## Inconsistencies
| Rule A | Rule B | Conflict | Severity | Resolution |

## Coverage Gaps
| Scenario | Why Gap | Impact | Recommendation |

## Ambiguities
| Rule | Ambiguity | Needs Clarification |

## Data Quality Issues
| Provider | Issue | Impact | Recommendation |

Be thorough."""

system_prompt = "Be comprehensive - include ALL rules. Do not summarize or abbreviate."

# Storage for results
results_haiku = {}
results_sonnet = {}

print("✓ Prompts defined and ready")
print(f"  Rules size: ~{estimate_tokens(rules):,} tokens")

### b. Run Haiku

In [ ]:
print("="*70)
print("RUNNING: HAIKU (3 TURNS)")
print("="*70)

# Setup
client = get_client("claude-haiku")
tracker = ExperimentTracker("haiku_run", budget_usd=1.00)
conversation_history = []

# Turn 1
print("\n[Haiku] Turn 1: General Patterns...")
response1 = client.invoke(prompt_turn_1, system=system_prompt, max_tokens=4096, temperature=0.0)
tracker.log(response1, notes="Turn 1")
conversation_history.append({"user": prompt_turn_1, "assistant": response1["content"]})
print(f"  ✓ {response1['output_tokens']:,} tokens | ${response1['cost']:.4f}")

# Turn 2
print("\n[Haiku] Turn 2: Provider-Specific...")
context = "\n\n---\n\n".join([f"User: {t['user']}\n\nAssistant: {t['assistant']}" for t in conversation_history])
response2 = client.invoke(f"{context}\n\n---\n\n{prompt_turn_2}", system=system_prompt, max_tokens=4096, temperature=0.0)
tracker.log(response2, notes="Turn 2")
conversation_history.append({"user": prompt_turn_2, "assistant": response2["content"]})
print(f"  ✓ {response2['output_tokens']:,} tokens | ${response2['cost']:.4f}")

# Turn 3
print("\n[Haiku] Turn 3: Analysis...")
context = "\n\n---\n\n".join([f"User: {t['user']}\n\nAssistant: {t['assistant']}" for t in conversation_history])
response3 = client.invoke(f"{context}\n\n---\n\n{prompt_turn_3}", system=system_prompt, max_tokens=4096, temperature=0.0)
tracker.log(response3, notes="Turn 3")
conversation_history.append({"user": prompt_turn_3, "assistant": response3["content"]})
print(f"  ✓ {response3['output_tokens']:,} tokens | ${response3['cost']:.4f}")

# Store results
results_haiku = {
    "model": "claude-haiku",
    "tracker": tracker,
    "responses": [response1, response2, response3],
    "conversation_history": conversation_history
}

print(f"\n✅ Haiku Complete: ${tracker.total_cost():.4f} | {tracker.total_tokens():,} tokens")


### c. Run Sonnet 3.5

In [ ]:
print("="*70)
print("RUNNING: SONNET 3.5 (3 TURNS)")
print("="*70)

# Setup
client = get_client("claude-sonnet-3.5")
tracker = ExperimentTracker("sonnet_run", budget_usd=2.00)
conversation_history = []

# Turn 1
print("\n[Sonnet] Turn 1: General Patterns...")
response1 = client.invoke(prompt_turn_1, system=system_prompt, max_tokens=8192, temperature=0.0)
tracker.log(response1, notes="Turn 1")
conversation_history.append({"user": prompt_turn_1, "assistant": response1["content"]})
print(f"  ✓ {response1['output_tokens']:,} tokens | ${response1['cost']:.4f}")

# Turn 2
print("\n[Sonnet] Turn 2: Provider-Specific...")
context = "\n\n---\n\n".join([f"User: {t['user']}\n\nAssistant: {t['assistant']}" for t in conversation_history])
response2 = client.invoke(f"{context}\n\n---\n\n{prompt_turn_2}", system=system_prompt, max_tokens=8192, temperature=0.0)
tracker.log(response2, notes="Turn 2")
conversation_history.append({"user": prompt_turn_2, "assistant": response2["content"]})
print(f"  ✓ {response2['output_tokens']:,} tokens | ${response2['cost']:.4f}")

# Turn 3
print("\n[Sonnet] Turn 3: Analysis...")
context = "\n\n---\n\n".join([f"User: {t['user']}\n\nAssistant: {t['assistant']}" for t in conversation_history])
response3 = client.invoke(f"{context}\n\n---\n\n{prompt_turn_3}", system=system_prompt, max_tokens=8192, temperature=0.0)
tracker.log(response3, notes="Turn 3")
conversation_history.append({"user": prompt_turn_3, "assistant": response3["content"]})
print(f"  ✓ {response3['output_tokens']:,} tokens | ${response3['cost']:.4f}")

# Store results
results_sonnet = {
    "model": "claude-sonnet-3.5",
    "tracker": tracker,
    "responses": [response1, response2, response3],
    "conversation_history": conversation_history
}

print(f"\n✅ Sonnet Complete: ${tracker.total_cost():.4f} | {tracker.total_tokens():,} tokens")


#### d. Compare Results & Save


In [ ]:
print("\n" + "="*70)
print("COMPARISON RESULTS")
print("="*70)

# Comparison table
comparison_data = {
    "Metric": [
        "Model",
        "Turn 1 Output",
        "Turn 2 Output", 
        "Turn 3 Output",
        "Total Tokens",
        "Total Cost",
        "Cost per 1K tokens"
    ],
    "Haiku": [
        results_haiku["model"],
        f"{results_haiku['responses'][0]['output_tokens']:,}",
        f"{results_haiku['responses'][1]['output_tokens']:,}",
        f"{results_haiku['responses'][2]['output_tokens']:,}",
        f"{results_haiku['tracker'].total_tokens():,}",
        f"${results_haiku['tracker'].total_cost():.4f}",
        f"${(results_haiku['tracker'].total_cost() / results_haiku['tracker'].total_tokens() * 1000):.4f}"
    ],
    "Sonnet 3.5": [
        results_sonnet["model"],
        f"{results_sonnet['responses'][0]['output_tokens']:,}",
        f"{results_sonnet['responses'][1]['output_tokens']:,}",
        f"{results_sonnet['responses'][2]['output_tokens']:,}",
        f"{results_sonnet['tracker'].total_tokens():,}",
        f"${results_sonnet['tracker'].total_cost():.4f}",
        f"${(results_sonnet['tracker'].total_cost() / results_sonnet['tracker'].total_tokens() * 1000):.4f}"
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print("\n" + df_comparison.to_string(index=False))

# Cost difference
haiku_cost = results_haiku['tracker'].total_cost()
sonnet_cost = results_sonnet['tracker'].total_cost()
cost_ratio = sonnet_cost / haiku_cost if haiku_cost > 0 else 0

print(f"\n💰 Cost Analysis:")
print(f"   Haiku: ${haiku_cost:.4f}")
print(f"   Sonnet: ${sonnet_cost:.4f}")
print(f"   Difference: ${sonnet_cost - haiku_cost:.4f} ({cost_ratio:.1f}x more expensive)")

# Save full documentation for both models
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for result in [results_haiku, results_sonnet]:
    model_name = result["model"]
    output_path = f"experiments/comprehensive_rules_{model_name}_{timestamp}.md"
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(f"# Bank Transaction Categorization Rules\n")
        f.write(f"## Generated by {model_name}\n\n")
        f.write(f"**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write(f"**Model**: {model_name}\n")
        f.write(f"**Cost**: ${result['tracker'].total_cost():.4f}\n")
        f.write(f"**Tokens**: {result['tracker'].total_tokens():,}\n\n")
        f.write("---\n\n")
        
        f.write("# 1. General Patterns\n\n")
        f.write(result['responses'][0]["content"])
        f.write("\n\n---\n\n")
        
        f.write("# 2. Provider-Specific\n\n")
        f.write(result['responses'][1]["content"])
        f.write("\n\n---\n\n")
        
        f.write("# 3. Analysis\n\n")
        f.write(result['responses'][2]["content"])
    
    print(f"\n✅ Saved: {output_path}")
    
    # Save experiment data
    result['tracker'].save()

# Save comparison report
comparison_path = f"experiments/comparison_report_{timestamp}.md"
with open(comparison_path, 'w', encoding='utf-8') as f:
    f.write("# Model Comparison Report\n\n")
    f.write(f"**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n")
    f.write("## Quantitative Comparison\n\n")
    f.write(df_comparison.to_markdown(index=False))
    f.write(f"\n\n## Cost Analysis\n\n")
    f.write(f"- **Haiku**: ${haiku_cost:.4f}\n")
    f.write(f"- **Sonnet 3.5**: ${sonnet_cost:.4f}\n")
    f.write(f"- **Difference**: ${sonnet_cost - haiku_cost:.4f} ({cost_ratio:.1f}x)\n\n")
    f.write("## Qualitative Assessment\n\n")
    f.write("Review the output files to assess:\n")
    f.write("- Column choices (Turn 1)\n")
    f.write("- Completeness (all turns)\n")
    f.write("- Detail level (Turn 2)\n")
    f.write("- Analysis depth (Turn 3)\n\n")
    f.write("## Recommendation\n\n")
    f.write("[Fill in after reviewing outputs]\n")

print(f"\n✅ Comparison report: {comparison_path}")

print("\n" + "="*70)
print("✅ COMPLETE!")
print("="*70)
print(f"📊 Compare outputs in experiments/ folder")
print(f"💡 Review both documentations and complete the comparison report")

---
## SECTION 9: EXAMPLE EXPERIMENTS

### Design Prompt

In [ ]:
# 1. Load the Confluence document
rules = load_confluence_doc("assets/bank_patterns.doc")

# 2. Check the size (optional but recommended)
tokens = context_summary(rules, "Confluence Rules")

# 3. Create client and tracker
client = get_client("claude-haiku")
tracker = ExperimentTracker("confluence_analysis", budget_usd=1.00)

# 4. Use with BOTH system and user prompts
response = client.invoke(
    prompt=f"""
Based on the business rules document below, please reorganize and document ALL the rules in a clear, structured format.

CRITICAL: This is a COMPREHENSIVE documentation task. Do NOT summarize or abbreviate. Include EVERY rule from the source document, even if the output is very long.

## Requirements

### 1. GENERAL PATTERNS
Start with rules that apply universally (across all providers):
- Separate into **SPEND** vs **NON-SPEND** categories
- For each rule, specify:
  - Pattern/condition (what triggers this rule)
  - base_category_type or category_key
  - Any amount thresholds or special conditions

### 2. PROVIDER-SPECIFIC PATTERNS
Then document rules that only apply to specific providers (e.g., ANZ, Macquarie, etc.):
- Group by provider
- For each rule, specify:
  - Provider name
  - Pattern/condition
  - base_category_type or category_key
  - Why this differs from the general pattern (if known)

### 3. FORMAT
Please use **tables** for clarity. Example format:

**General Spend Patterns:**
| Pattern/Condition | Category Key | Notes |
|-------------------|--------------|-------|
| Merchant contains "WOOLWORTHS" | GROCERIES | ... |

**Provider-Specific Patterns (ANZ):**
| Pattern/Condition | Category Key | Differs From General | Notes |
|-------------------|--------------|---------------------|-------|
| ... | ... | ... | ... |

### 4. INCONSISTENCIES & GAPS
At the end, add a section that flags:
- **Inconsistencies**: Rules that conflict or contradict each other
- **Gaps**: Scenarios not covered by any rule
- **Ambiguities**: Rules that are unclear or could be interpreted multiple ways

## Important
- Be COMPREHENSIVE - include ALL rules from the document
- Do NOT summarize or condense - this is complete documentation
- If a rule is unclear, include it anyway and flag it
- Preserve the original category_key/base_category_type names exactly as written

=== BUSINESS RULES DOCUMENT ===
{rules}
=== END DOCUMENT ===
""",
    system="You are a financial transaction categorization expert. Be detailed and structured in your analysis, we want to comprehensively explain the logic and make it easier for users to understand",
    max_tokens=4096,
    temperature=0.0
)

# 5. Log and view results
tracker.log(response, prompt_preview="Analyze Confluence rules", notes="Initial analysis")

print("\n" + "="*70)
print("ANALYSIS RESULTS")
print("="*70)
print(response["content"])
print(f"\nTokens used: {response['input_tokens']} in / {response['output_tokens']} out")
print(f"Cost: ${response['cost']:.6f}")

# 6. Save experiment
tracker.save()

In [ ]:
# Save output to file for reference
output_path = f"experiments/restructured_rules_{response['model']}_{tracker.timestamp}.md"
with open(output_path, "w") as f:
    f.write(response["content"])
print(f"\nSaved to: {output_path}")

In [ ]:
# 1. Load the Confluence document
rules = load_confluence_doc("assets/bank_patterns.doc")

# 2. Check the size (optional but recommended)
tokens = context_summary(rules, "Confluence Rules")

# 3. Create client and tracker
client = get_client("claude-sonnet-3.5")
tracker = ExperimentTracker("confluence_analysis", budget_usd=1.00)

# 4. Use with BOTH system and user prompts
response = client.invoke(
    prompt=f"""
Based on the business rules document below, please reorganize and document ALL the rules in a clear, structured format.

CRITICAL: This is a COMPREHENSIVE documentation task. Do NOT summarize or abbreviate. Include EVERY rule from the source document, even if the output is very long.

## Requirements

### 1. GENERAL PATTERNS
Start with rules that apply universally (across all providers):
- Separate into **SPEND** vs **NON-SPEND** categories
- For each rule, specify:
  - Pattern/condition (what triggers this rule)
  - base_category_type or category_key
  - Any amount thresholds or special conditions

### 2. PROVIDER-SPECIFIC PATTERNS
Then document rules that only apply to specific providers (e.g., ANZ, Macquarie, etc.):
- Group by provider
- For each rule, specify:
  - Provider name
  - Pattern/condition
  - base_category_type or category_key
  - Why this differs from the general pattern (if known)

### 3. FORMAT
Please use **tables** for clarity. Example format:

**General Spend Patterns:**
| Pattern/Condition | Category Key | Notes |
|-------------------|--------------|-------|
| Merchant contains "WOOLWORTHS" | GROCERIES | ... |

**Provider-Specific Patterns (ANZ):**
| Pattern/Condition | Category Key | Differs From General | Notes |
|-------------------|--------------|---------------------|-------|
| ... | ... | ... | ... |

### 4. INCONSISTENCIES & GAPS
At the end, add a section that flags:
- **Inconsistencies**: Rules that conflict or contradict each other
- **Gaps**: Scenarios not covered by any rule
- **Ambiguities**: Rules that are unclear or could be interpreted multiple ways

## Important
- Be COMPREHENSIVE - include ALL rules from the document
- Do NOT summarize or condense - this is complete documentation
- If a rule is unclear, include it anyway and flag it
- Preserve the original category_key/base_category_type names exactly as written

=== BUSINESS RULES DOCUMENT ===
{rules}
=== END DOCUMENT ===
""",
    system="""You are a financial transaction categorization expert. Be detailed and structured in your analysis, we want to comprehensively explain 
    the logic and make it easier for users to understand. i want to include the following: 
    - Provider-specific patterns
    - Special transaction types
    - Category-specific rules
    - Inconsistencies & gaps"""
  ,
    max_tokens=4096,
    temperature=0.0
)

# 5. Log and view results
tracker.log(response, prompt_preview="Analyze Confluence rules", notes="Initial analysis")

print("\n" + "="*70)
print("ANALYSIS RESULTS")
print("="*70)
print(response["content"])
print(f"\nTokens used: {response['input_tokens']} in / {response['output_tokens']} out")
print(f"Cost: ${response['cost']:.6f}")

# 6. Save experiment
tracker.save()

In [ ]:
# Save output to file for reference
output_path = f"experiments/restructured_rules_{response['model']}_{tracker.timestamp}.md"
with open(output_path, "w") as f:
    f.write(response["content"])
print(f"\nSaved to: {output_path}")

---
## SECTION 10: TROUBLESHOOTING GUIDE

### 🔧 Common Issues and Solutions

**Problem: "AccessDeniedException: bedrock:InvokeModel"**
- **Cause**: Missing IAM permissions
- **Solution**:
  1. Check IAM role has `bedrock:InvokeModel` permission
  2. Verify using SageMaker execution role
  3. Contact DevOps to add permissions

**Problem: "ThrottlingException"**
- **Cause**: Exceeded rate limits
- **Solution**:
  - Framework auto-retries 3 times
  - Add delays: `time.sleep(1)` between requests
  - Request quota increase in AWS Console

**Problem: "ValidationException: max_tokens"**
- **Cause**: Exceeds model's output limit
- **Solution**:
  - Framework auto-caps to model limit
  - Use `client.get_output_limit()` to check
  - Switch to Sonnet 3.5 for 8K output

**Problem: Budget Exceeded**
- **Cause**: Hit spending limit
- **Solution**:
  - Increase budget: `ExperimentTracker(budget_usd=2.00)`
  - Review costs: `tracker.summary()`
  - Use cheaper model: Switch to Haiku

**Problem: Response is truncated**
- **Cause**: Hit max_tokens limit
- **Solution**:
  - Check if `output_tokens == max_tokens`
  - Increase max_tokens or use Sonnet 3.5
  - Ask for shorter response in prompt

---
## SECTION 11: REVIEW & SAVE EXPERIMENTS

In [ ]:
# View all experiments in this session
df = tracker.summary()
df

In [ ]:
# Save experiments to CSV
tracker.save()

---
## QUICK REFERENCE

### Models
```python
client = get_client("claude-haiku")      # Cheapest Claude
client = get_client("claude-sonnet-3.5") # Best Claude
client = get_client("nova-lite")         # Cheap Nova
```

### Context Limits
| Model | Context (Input) | Max Output |
|-------|----------------|------------|
| Claude Haiku | 200K tokens | 4K tokens |
| Claude Sonnet 3.5 | 200K tokens | 8K tokens |
| Nova Lite/Pro | 300K tokens | 5K tokens |

### Cost (per 1M tokens)
| Model | Input | Output |
|-------|-------|--------|
| Claude Haiku | $0.25 | $1.25 |
| Nova Lite | $0.06 | $0.24 |

### Workflow
```python
# 1. Load context
rules = load_markdown_file("path/to/file.md")

# 2. Create client & tracker
client = get_client("claude-haiku")
tracker = ExperimentTracker("test", budget_usd=1.00)

# 3. Run experiment
response = client.invoke(prompt, max_tokens=4096, temperature=0.0)

# 4. Log & save
tracker.log(response, notes="Test run")
tracker.save()
```